In [7]:
import pandas as pd
import os
import h5py
import pandas as pd



## DATASET SAMO SA PESMAMA I METAPODACIMA

In [10]:
BASE_DIR = './MillionSongSubset'


sve_pesme = []
brojac_gresaka = 0
print("kreni")


for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file.endswith('.h5'):
            file_path = os.path.join(root, file)
       
            try:
                with h5py.File(file_path, 'r') as f:
                    meta_songs = f['metadata']['songs'][0]
                    raw_terms = f['metadata']['artist_terms'][:]
                    raw_artist_similarity = f['metadata']['similar_artists'][:]
                    raw_terms_weights = f['metadata']['artist_terms_weight'][:]
                    
                    song_id = meta_songs['song_id'].decode('utf-8', errors='replace')
                    artist_id =  meta_songs['artist_id'].decode('utf-8', errors='replace')
                    artist_terms = [term.decode('utf-8') for term in raw_terms]
                    artist_name = meta_songs['artist_name'].decode('utf-8', errors='replace')
                    artist_terms_weights = [weight for weight in raw_terms_weights]
                    analysis_songs = f['analysis']['songs'][0]
                    loudness = analysis_songs['loudness']
                    tempo = analysis_songs['tempo']
                    sim_art = [f.decode('utf-8') for f in raw_artist_similarity]
                    modes = int(analysis_songs['mode'])
                    mb_songs = f['musicbrainz']['songs'][0]
                    timesig = analysis_songs['time_signature']  
                    title = meta_songs['title'].decode('utf-8', errors='replace')
                    sve_pesme.append({

                       'song_id': song_id,
                        'artist_id': artist_id,
                        'title' : title,
                        'artist_name': artist_name,
                        'artist_terms': artist_terms,
                        'artist_terms_weight': artist_terms_weights,
                        'loudness': loudness,
                        'tempo': tempo,
                       'similar_artists': sim_art,
                        'mode': modes,
                        'time_signature': timesig
                    })
            except Exception as e:
                brojac_gresaka += 1
                if brojac_gresaka <= 3:
                    print(f"{file}: {e}")

print(f"grešaka: {brojac_gresaka}")

df = pd.DataFrame(sve_pesme)
print( {len(df)} )

if not df.empty:
    display(df.head())

kreni
grešaka: 0
{10000}


,song_id,artist_id,title,artist_name,artist_terms,artist_terms_weight,loudness,tempo,similar_artists,mode,time_signature
0,SOMZWCG12A8C13C480,ARD7TVE1187B99BFB1,I Didn't Mean To,Casual,"[hip hop, underground rap, g funk, alternative...","[1.0, 0.8979359555142553, 0.8842618474718359, ...",-11.197,92.198,"[ARV4KO21187FB38008, ARWHM281187FB3D381, ARJGO...",0,4
1,SOCIWDW12A8C13D406,ARMJAGH1187FB546F3,Soul Deep,The Box Tops,"[blue-eyed soul, pop rock, blues-rock, beach m...","[1.0, 0.8459884034332037, 0.8306895698215381, ...",-9.843,121.274,"[ARSZWK21187B9B26D7, ARLDW2Y1187B9B544F, ARG0T...",0,4
2,SOXVLOJ12AB0189215,ARKRRTF1187B9984DA,Amor De Cabaret,Sonora Santanera,"[salsa, cumbia, tejano, ranchera, latin pop, l...","[1.0, 0.9582578450180738, 0.9582578450180738, ...",-9.689,100.070,"[ARFSJUG11C8A421AAD, AR8SD041187FB36015, ARR75...",1,1
3,SONHOTT12A8C13493C,AR7G5I41187FB4CE6C,Something Girls,Adam Ant,"[pop rock, new wave, dance rock, rock, new rom...","[1.0, 0.9636972066614938, 0.9267729972686404, ...",-9.013,119.293,"[AR4R0741187FB39AF2, AR0D7K21187B9AD14E, ARRCB...",1,4
4,SOFSOCN12A8C143F5D,ARXR32B1187FB57099,Face the Ashes,Gob,"[pop punk, ska punk, breakcore, alternative me...","[1.0, 0.9609057433767874, 0.9592366232787087, ...",-4.501,129.738,"[ARUA62A1187B99D9B0, ARHJFFY1187B98BA76, ARHB1...",1,4


## MERGE

#### sa metapodacima(similar_artists, time_signature, tempo, mode, loudness, artist_terms, artist_id, song_id, play_count, user_id)

In [11]:
merge = pd.read_csv('mergeovan.csv')
merge.head()

,user_id,song_id,play_count,artist_name,artist_id,artist_terms,loudness,mode,tempo,time_signature,similar_artists
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOWEZSI12A81C21CE6,1,NaN,AR2UQQ51187B9AC816,"['flamenco', 'soundtrack', 'folk', 'spanish', ...",0.828210,0.0,0.627810,0.142857,"['AR9Z7JB1187B99DB3D', 'ARC1SF21187FB51D0F', '..."
1,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SODCXXY12AB0187452,2,NaN,ARXLMH011C8A415658,"['pop rap', 'crunk', 'rapcore', 'screamo', 'br...",0.767205,1.0,0.455096,0.571429,"['ARHAUVU122BCFCBA38', 'AR258TI11C8A416B5B', '..."
2,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SOWPAXV12A67ADA046,18,NaN,ARUQ6301187FB54EBA,"['pop rap', 'hip hop', 'hip house', 'new jack ...",0.880595,1.0,0.485477,0.571429,"['ARNHMFD1187FB3B3F6', 'ARELPXQ1187FB384FD', '..."
3,b64cdd1a0bd907e5e00b39e345194768e330d652,SOLXDDC12A6701FBFD,1,NaN,ARTH9041187FB43E1F,"['hip hop', 'rap', 'hardcore rap', 'club', 'so...",0.912755,0.0,0.685498,0.571429,"['AR23C041187FB4D534', 'ARZER7I1187FB385AF', '..."
4,b64cdd1a0bd907e5e00b39e345194768e330d652,SONJBQX12A6D4F8382,4,NaN,ARF8HTQ1187B9AE693,"['techno', 'electronica', 'electronic', 'pop',...",0.893026,0.0,0.423094,0.571429,"['AR3NPVS1187FB5108F', 'ARMKBL21187FB38230', '..."


## TRAIN DS

In [12]:
df_train = pd.read_csv('df_train.csv')
df_train.head()

,user_id,song_id,play_count,artist_id,artist_terms,loudness,tempo,time_signature,similar_artists
0,bn,SOSKFIT12A8C14224A,1,ARM7EDF1187B9B3FA1,"['heavy metal', 'hard rock', 'rock', 'metal', ...",0.914632,0.510554,1.000000,"['ARE8GLF1187FB52532', 'ARKUDKL1296AAF0B2F', '..."
1,2000s pop,SOPVXLX12A8C1402D5,1,AR3JMC51187B9AE49D,"['teen pop', 'pop', 'rock', 'adult contemporar...",0.891092,0.429129,0.571429,"['AR03BDP1187FB5B324', 'AR5YRCN1187B98E3C8', '..."
2,60's,SONFECQ12AB017EFF7,1,AROCBZZ11E2835D652,"[""rock 'n roll"", 'blues-rock', 'british invasi...",0.823613,0.469436,0.571429,"['ARQNR1G1187B9AE88F', 'ARPISA41187FB3DE42', '..."
3,60's,SONFECQ12AB017EFF7,1,AROCBZZ11E2835D652,"[""rock 'n roll"", 'blues-rock', 'british invasi...",0.823613,0.469436,0.571429,"['ARQNR1G1187B9AE88F', 'ARPISA41187FB3DE42', '..."
4,60's,SONFECQ12AB017EFF7,1,AROCBZZ11E2835D652,"[""rock 'n roll"", 'blues-rock', 'british invasi...",0.823613,0.469436,0.571429,"['ARQNR1G1187B9AE88F', 'ARPISA41187FB3DE42', '..."


## TRAIN DS VECI 

In [ ]:
df_trainveci = pd.read_csv('df_trainveci.csv')
df_trainveci.head()

## TEST DS

In [ ]:
df_test = pd.read_csv('df_test.csv')
df_test.head()

## VALIDATION DS

In [ ]:
df_val = pd.read_csv('df_val.csv')
df_val.head()

## SPOJENO

#### users + playliste

In [16]:
df_spojen= pd.read_csv('filtriran_dataset.csv')
df_spojen.head()

,user_id,song_id,play_count,artist_name
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOWEZSI12A81C21CE6,1,NaN
1,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SODCXXY12AB0187452,2,NaN
2,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SOWPAXV12A67ADA046,18,NaN
3,b64cdd1a0bd907e5e00b39e345194768e330d652,SOLXDDC12A6701FBFD,1,NaN
4,b64cdd1a0bd907e5e00b39e345194768e330d652,SONJBQX12A6D4F8382,4,NaN
